In [ ]:
from copy import deepcopy
import os

import numpy as np
# import pandas as pd
# from matplotlib import pyplot as plt
# import fastplotlib as fpl

import mesmerize_core as mc
# import mesmerize_viz

import glob
# import scipy.io
# import h5py

In [ ]:
# from mesmerize_core.caiman_extensions.cnmf import cnmf_cache

In [ ]:
# if os.name == "nt":
#     # disable the cache on windows, this will be automatic in a future version
#     cnmf_cache.set_maxsize(0)

In [ ]:
mc.set_parent_raw_data_path('/home/wanglab/data/2P/')

In [ ]:
data_path = mc.get_parent_raw_data_path().joinpath('C57_O1M2/TSeries-10022023-1300-001')

In [ ]:
# To concatenate ome.tif files to hdf5 (deprecated)
# chain_tiff = True
# data_path = 'D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001'
# export_path = mc.get_parent_raw_data_path().joinpath('export')
# import caiman as cm
# if chain_tiff:
#     # Start the timer
#     start_time = time.time()
    
#     fls = glob.glob(os.path.join(data_path,'*_Ch2_*.ome.tif'))

# fls = (glob.glob(os.path.join(sourcedir1, '*_Ch2_*.ome.tif')) +
#        glob.glob(os.path.join(sourcedir2, '*_Ch2_*.ome.tif')) +
#        glob.glob(os.path.join(sourcedir3, '*_Ch2_*.ome.tif')) +
#        glob.glob(os.path.join(sourcedir4, '*_Ch2_*.ome.tif')) +
#        glob.glob(os.path.join(sourcedir5, '*_Ch2_*.ome.tif')) +
#        glob.glob(os.path.join(sourcedir6, '*_Ch2_*.ome.tif')))

#     fls.sort()  # make sure your files are sorted alphanumerically
    
#     # Count the files
#     file_count = len(fls)
#     # print(f"Found {file_count} files to concatenate.")

    # m = cm.load_movie_chain(fls,outtype=np.ushort)
    # m.save(os.path.join(export_path,'data.h5'),to32=False)

    # # Calculate the elapsed time
    # elapsed_time = time.time() - start_time

In [ ]:
# print(f"Found {file_count} files to concatenate.")
# print(f"Concatenation completed in {elapsed_time:.2f} seconds.")

In [ ]:
# Just concatenate tiff files - works nad opens in ImageJ, but fails with caiman
# WARNING:tifffile.tifffile:<tifffile.read_uic_tag> invalid string in tag 'CalibrationUnits'
# WARNING:tifffile.tifffile:<tifffile.read_uic_tag> invalid string in tag 'Name'
# WARNING:tifffile.tifffile:<tifffile.read_uic_tag> invalid string in tag 'CalibrationUnits'
# WARNING:tifffile.tifffile:<tifffile.read_uic_tag> invalid string in tag 'Name'
# WARNING:tifffile.tifffile:<tifffile.read_uic_tag> failed reading CreateTime with ValueError: no datetime before year 1 (julianday=0)
# WARNING:tifffile.tifffile:<tifffile.read_uic_tag> failed reading LastSavedTime with ValueError: no datetime before year 1 (julianday=0)
# ...

# import glob
# import os
# import time
# from tifftools import tiff_concat

# def concat_tiff_files(source_dir, output_file):
#     # Start the timer
#     start_time = time.time()

#     # Create a list of TIFF files sorted by name
#     tiff_files = sorted(glob.glob(os.path.join(source_dir, '*_Ch2_*.ome.tif')))
    
#     # Count the files
#     file_count = len(tiff_files)
#     print(f"Found {file_count} files to concatenate.")

#     # Concatenate the files
#     tiff_concat(tiff_files, output_file, overwrite=True)

#     # Calculate the elapsed time
#     elapsed_time = time.time() - start_time
#     print(f"Concatenation completed in {elapsed_time:.2f} seconds.")

# # Define your paths
# # data_path = 'D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001'
# # export_path = mc.get_parent_raw_data_path().joinpath('export')
# export_path = data_path

# # Call the function
# concat_tiff_files(data_path, os.path.join(export_path, 'alltiffs.tiff'))

In [ ]:
# Variant of concat_tiff_files for ome.tif files, using tifffile, to standardize or strip the metadata before concatenation. 

import glob
import os
import time
from tifftools import tiff_concat
from tifffile import imread, imwrite, TiffWriter, TiffFile
import numpy as np

def concat_tiff_files(file_list, output_file):
    # Concatenate the files
    tiff_concat(file_list, output_file, overwrite=True)
    print(f"Concatenated {len(file_list)} files to {output_file}.")
    return

def strip_metadata(concat_tiffs, output_file):
    """ Strip metadata from a tiff stack. """
    import warnings 
    # print(f"Info about {concat_tiffs}:")
    # with TiffFile(concat_tiffs) as tf:
    #     num_images = len(tf.pages)
    #     first_ifd = tf.pages[0]  # image file directory
    #     is_imagej = tf.is_imagej
    #     is_bigtiff = tf.is_bigtiff
    # im_dtype = first_ifd.dtype
    # im_shape = first_ifd.shape
    # print(f"\tNumber of images: {num_images}")
    # print(f"\tDimensions of first image: {im_shape}")
    # print(f"\tByte representation: {im_dtype}")
    # print(f"\tImageJ format? {is_imagej}")
    # print(f"\tBigTIFF format? {is_bigtiff}")
    
    with warnings.catch_warnings():
        # ignore "*invalid string in tag 'CalibrationUnits'*" and "*invalid string in tag 'Name'*"
        # warnings.simplefilter("ignore", UserWarning)

        with TiffFile(concat_tiffs) as tif:
            # First method below creates a single page tiff file and caiman ultimately fails 
            # # Create an empty list to store each frame
            # frames = []
            # for page in tif.pages:
            #     # Append each frame to the list
            #     frames.append(page.asarray())
            # # Stack all frames into a single array
            # tiff_stack = np.stack(frames, axis=0)
            # print(f"Number of frames: {tiff_stack.shape[0]}")
            # # Now tiff_stack contains all frames as separate entities
            
            # # Save the stripped metadata TIFF
            # imwrite(output_file, tiff_stack, imagej=False, bigtiff=True)
            # print(f"Stripped metadata from {concat_tiffs} and saved to {output_file}.")    
    
            # Instead write a multi-page TIFF

            images = [page.asarray() for page in tif.pages]

            with TiffWriter(output_file, bigtiff=True) as new_tif:
                for img in images:
                    new_tif.write(img, photometric='minisblack')

            print(f"Multi-page TIFF saved to {output_file}")
    
    return
    
# Define your paths
# data_path = 'D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001'
# export_path = mc.get_parent_raw_data_path().joinpath('export')
export_path = data_path  
# Create a list of TIFF files sorted by name
tiff_files = sorted(glob.glob(os.path.join(data_path,'*_Ch2_*.ome.tif')))
# Output file
output_file = os.path.join(export_path, 'concatenated_tiff.tiff')

# Call the function
concat_tiff_files(tiff_files, output_file)

# Concatenate the ome.tif files
fixed_file = os.path.join(export_path, 'fixedtiff.tiff')
strip_metadata(output_file, fixed_file)

In [ ]:
# data_path = 'D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001'
movie_path = mc.get_parent_raw_data_path().joinpath('C57_O1M2/TSeries-10022023-1300-001', 'fixedtiff.tiff')
# alltiffs
# fixedtiff
movie_path

In [ ]:
batch_path = mc.get_parent_raw_data_path().joinpath("mesmerize-batch/batch_new.pickle")
batch_path

In [ ]:
df = mc.create_batch(batch_path)
# df = mc.load_batch(batch_path)

In [ ]:
# We will start with one version of parameters
mcorr_params1 =\
{
  'main': # this key is necessary for specifying that these are the "main" params for the algorithm
    {
        'max_shifts': [24, 24],
        'strides': [48, 48],
        'overlaps': [24, 24],
        'max_deviation_rigid': 3,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

In [ ]:
# add an item to the DataFrame
df.caiman.add_item(
    algo='mcorr',
    input_movie_path=movie_path,
    params=mcorr_params1,
    item_name=movie_path.stem,  # filename of the movie, but can be anything
)

df

In [ ]:
# small fix for CONDA_PREFIX_1 not being found in environment
import os
str_to_remove = os.path.join(' ', 'envs', 'mescore')[1:]
conda_prefix_1_str = os.environ['CONDA_PREFIX'].replace(str_to_remove, '')
os.environ['CONDA_PREFIX_1'] = conda_prefix_1_str

In [ ]:
df.iloc[0].caiman.run()

In [ ]:
df.iloc.

### Reload DataFrame from disk

After running one or any number of items in `mesmerize-core` you must call `df = df.caiman.reload_from_disk()`. This loads the DataFrame with references to the output files in the batch directory.

In [ ]:
df = df.caiman.reload_from_disk()

In [ ]:
df

### Outputs

We can see that the outputs column has been populated. The entries in this column do not have to be accessed directly. The `mesmerize-core` API allows you to fetch these outputs.

In [ ]:
index = 0 # we will fetch stuff from index 0 which we just ran

# get the motion corrected movie memmap
mcorr_movie = df.iloc[0].mcorr.get_output()
mcorr_movie.shape

In [ ]:
# path to the mcorr memmap if you ever need it
mcorr_memmap_path = df.iloc[0].mcorr.get_output_path()
mcorr_memmap_path

In [ ]:
# mean projection, max and std projections are also available
mean_proj = df.iloc[0].caiman.get_projection("mean")
mean_proj.shape

In [ ]:
# the input movie, note that we use `.caiman` here instead of `.mcorr`
input_movie = df.iloc[0].caiman.get_input_movie()
input_movie

## Note on input movies

`get_input_movie()` will automatically work with most tiff files and memmaps. If you want to load other file types, you will need to pass it a function (see examples below) that returns a lazy-loadable array, or a numpy array if you have enough RAM. 


### tiff files

`get_input_movie()` wil try to use `tifffile.memmap` to lazy-load tiff files. This works for some tiff files. If `tifffile.memmap` fails, `mesmerize-core` has its own `LazyTiff` implementation that it will try to fallback on. However, some not every tiff file can be lazy-loaded so it is not gaurenteed that `LazyTiff` will work. The implementation of `LazyTiff` is quite simple and you might be able to subclass it for your specific type of tiff file: https://github.com/nel-lab/mesmerize-core/blob/master/mesmerize_core/arrays/_tiff.py 

### hdf5 files

```python
import h5py

def hdf5_reader(path):
    f = h5py.File(path)
    return f["your-key"]

input_movie = df.iloc[index].caiman.get_input_movie(hdf5_reader)
```

### avi files
    
```python
from mesmerize_core.arrays import LazyVideo

def video_reader(path):
    a = LazyVideo(path)  # you can use the other args if you want
    return a

input_movie = df.iloc[index].caiman.get_input_movie(video_reader)
```